# FLAN-T5-Base LoRA Fine-tuning for Text Simplification

This notebook fine-tunes `google/flan-t5-base` (248M params) with LoRA on the
`combined_simplified_19830.json` dataset using a 70:30 train/test split.

**Key fixes applied:**
- `fp16=False` to prevent NaN / 0.0 loss on T5 layer-norm
- No static padding in tokenizer (delegated to `DataCollatorForSeq2Seq` for proper `-100` label masking)
- Beam search + repetition penalty + n-gram blocking in inference to prevent sentence looping
- `save_strategy='epoch'` without conflicting `save_steps`

In [ ]:
!pip install -q transformers datasets peft accelerate sentencepiece

In [ ]:
import os
import torch
from google.colab import drive
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)
from peft import get_peft_model, LoraConfig, TaskType, PeftModel

drive.mount('/content/drive')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'CUDA available: {torch.cuda.is_available()} | Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Load the 19,830-row dataset directly from the GitHub repo
dataset_url = 'https://raw.githubusercontent.com/Pedri-8/Research-Intern/main/output/combined_simplified_19830.json'
print(f'Loading dataset from: {dataset_url}')

raw_datasets = load_dataset('json', data_files=dataset_url)

# 70% train, 30% test
split_dataset = raw_datasets['train'].train_test_split(test_size=0.3, seed=42)
print(split_dataset)

In [ ]:
# Initialize model and tokenizer
model_name = 'google/flan-t5-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.to(device)
print(f'Model: {model_name} loaded on {device}')

In [ ]:
# Tokenize — NO padding here; DataCollatorForSeq2Seq will handle it
# and properly mask pad tokens to -100 in labels
max_input_length = 512
max_target_length = 256

def preprocess_function(examples):
    inputs = [f'Simplify the following text:\n{txt}' for txt in examples['text']]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)
    labels = tokenizer(text_target=examples['simplified'], max_length=max_target_length, truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized_datasets = split_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=split_dataset['train'].column_names,
)

print(f'Train examples: {len(tokenized_datasets["train"])}')
print(f'Test examples:  {len(tokenized_datasets["test"])}')

In [ ]:
# Configure LoRA adapter
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=['q', 'v'],
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# Training configuration
training_args = Seq2SeqTrainingArguments(
    output_dir='/content/drive/MyDrive/flan_t5_lora_simplify_base',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=1e-4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=5,
    logging_steps=50,
    predict_with_generate=True,
    fp16=False,                   # FP32 to prevent T5 NaN/0.0 loss
    load_best_model_at_end=True,
    metric_for_best_model='loss',
    report_to='none',
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    data_collator=data_collator,
)

print('Starting training...')
trainer.train()

In [ ]:
# Save the LoRA adapter to Google Drive
adapter_dir = '/content/drive/MyDrive/flan_t5_lora_adapter_base'
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f'LoRA adapter saved to: {adapter_dir}')

In [ ]:
# Merge adapter into base model and save the full standalone model
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel

model_name = 'google/flan-t5-base'
adapter_dir = '/content/drive/MyDrive/flan_t5_lora_adapter_base'
merged_dir = '/content/drive/MyDrive/flan_t5_lora_merged_base'

tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
peft_model = PeftModel.from_pretrained(base_model, adapter_dir)
merged_model = peft_model.merge_and_unload()

merged_model.save_pretrained(merged_dir)
tokenizer.save_pretrained(merged_dir)
print(f'Full merged model saved to: {merged_dir}')

In [ ]:
# Inference test with repetition-safe generation
sample_text = (
    'Wellington had deployed them on the reverse slope of the ridge, '
    'where they could neither be easily seen nor easily softened up with artillery.'
)

prompt = f'Simplify the following text:\n{sample_text}'
inputs = tokenizer(prompt, return_tensors='pt', max_length=512, truncation=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

merged_model.to(device)
with torch.no_grad():
    outputs = merged_model.generate(
        **inputs,
        max_new_tokens=256,
        num_beams=4,
        repetition_penalty=1.5,
        no_repeat_ngram_size=3,
        early_stopping=True,
    )

print(f'Original:   {sample_text}')
print(f'Simplified: {tokenizer.decode(outputs[0], skip_special_tokens=True)}')